# Lesson 2 ? Typed Structured Renewal Review

Convert a messy fictional account-manager note into a Pydantic RenewalReview with client.responses.parse(..., text_format=RenewalReview) and response.output_parsed.

Schema adherence controls shape, not truth. This lesson verifies money and percentage fields against deterministic Harborlight services.

In [ ]:
import os

import pandas as pd
from openai import OpenAI

from harborlight_responses.config import api_key_available, resolve_model
from harborlight_responses.fixtures import FIXTURE_LABEL, STRUCTURED_REVIEW_FIXTURE
from harborlight_responses.services import verify_review_arithmetic
from harborlight_responses.structured import parse_renewal_review

note = """
Renewal note ? all details are fictional.
Beacon Books / FIC-HLA-1002 renews 2026-07-10.
Current premium is 126000 cents and proposal is 132300 cents.
The owner asked for a plain-language explanation and a follow-up before renewal.
Treat this as review attention, not an urgent underwriting decision.
""".strip()

selection = resolve_model()
live_requested = os.getenv("HARBORLIGHT_LIVE") == "1"
live = live_requested and api_key_available()
print("Selected model:", selection.model)
print("Mode:", "Live API" if live else "Demo Fixture")
print("Messy note:")
print(note)

In [ ]:
if live:
    client = OpenAI()
    raw_response, review = parse_renewal_review(
        client,
        model=selection.model,
        note=note,
    )
    print("Typed result source: response.output_parsed")
else:
    review = STRUCTURED_REVIEW_FIXTURE
    print(FIXTURE_LABEL)

verified_change = verify_review_arithmetic(review)
print()
print("Typed object:")
print(review)
print()
print("Deterministic arithmetic check:")
print(verified_change)

table = pd.DataFrame([review.model_dump(mode="json")])
table

## Refusals and missing results

Production code must not assume output_parsed exists. parsed_renewal_review(...) checks for refusal content, rejects a missing parsed object, validates mappings as RenewalReview, and verifies premium arithmetic. Structured output is safer than parsing prose because field names, types, enums, and extra-field behavior are explicit?but human review and deterministic checks still matter.